### 登录

In [1]:
import requests
import json
from os.path import expanduser
from requests.auth import HTTPBasicAuth
import pandas as pd

with open(expanduser(r"C:\Users\nay\Desktop\qr\qr\worldquant\idcode.txt")) as f:
    credentials = json.load(f)

username,password = credentials
sess = requests.Session()
sess.auth = HTTPBasicAuth(username, password)
response = sess.post('https://api.worldquantbrain.com/authentication')
print(response.status_code)
print(response.json())

201
{'user': {'id': 'LH74870'}, 'token': {'expiry': 14400.0}, 'permissions': ['TUTORIAL']}


### 获取数据集id为fundamental6（company fundamental data for equity）下的所有数据字段

In [2]:
def get_datafields(
        s,
        searchScope,
        dataset_id:str = '',
        search: str = ''
):
    import pandas as pd
    instrument_type = searchScope['instrumentType']
    region = searchScope['region']
    delay = searchScope['delay']
    universe = searchScope['universe']
    if len(search) == 0:
        url_template = 'https://api.worldquantbrain.com/data-fields?' +\
            f"instrumentType={instrument_type}&region={region}&delay={delay}&universe={universe}&dataset.id={dataset_id}&limit=50"+\
            "&offset={x}"
        count = s.get(url_template.format(x=0)).json()['count']
    else:
        url_template = 'https://api.worldquantbrain.com/data-fields/search?' +\
            f"instrumentType={instrument_type}"+\
            f"&region={region}&delay={str(delay)}&universe={universe}&dataset.id={dataset_id}&limit=50"+\
            f"&search={search}"+\
            "&offset={x}"
        count = 100

    datafields_list = []
    for x in range(0, count, 50):
        datafields = s.get(url_template.format(x=x))
        datafields_list.append(datafields.json()['results'])
    datafields_list_flat = [item for sublist in datafields_list for item in sublist]
    datafields_df = pd.DataFrame(datafields_list_flat)
    return datafields_df

In [3]:
searchscope = {'region':'USA','delay':'1','universe':'TOP3000','instrumentType':'EQUITY'}
fundamental6 = get_datafields(s=sess,searchScope=searchscope,dataset_id='fundamental6')


In [4]:
fundamental6 = fundamental6[fundamental6['type']=='MATRIX']
fundamental6.head()

,id,description,dataset,category,subcategory,region,delay,universe,type,coverage,userCount,alphaCount,themes
0,assets,Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,0.5,34199,118182,[]
1,assets_curr,Current Assets - Total,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,0.5,2906,13718,[]
2,bookvalue_ps,Book Value Per Share,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,0.5,2258,9138,[]
3,capex,Capital Expenditures,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,0.5,9725,24016,[]
4,cash,Cash,"{'id': 'fundamental6', 'name': 'Company Fundam...","{'id': 'fundamental', 'name': 'Fundamental'}","{'id': 'fundamental-fundamental-data', 'name':...",USA,1,TOP3000,MATRIX,0.5,2096,10914,[]


In [7]:
datafields_list_fundamental6 = fundamental6['id'].values
datafields_list_fundamental6

array(['assets', 'assets_curr', 'bookvalue_ps', 'capex', 'cash',
       'cash_st', 'cashflow', 'cashflow_dividends', 'cashflow_fin',
       'cashflow_invst', 'cashflow_op', 'cogs', 'current_ratio', 'debt',
       'debt_lt', 'debt_st', 'depre_amort', 'ebit', 'ebitda', 'employee',
       'enterprise_value', 'eps', 'equity', 'fnd6_acdo', 'fnd6_acodo',
       'fnd6_acox', 'fnd6_acqgdwl', 'fnd6_acqintan',
       'fnd6_adesinda_curcd', 'fnd6_aldo', 'fnd6_am', 'fnd6_aodo',
       'fnd6_aox', 'fnd6_aqc', 'fnd6_aqi', 'fnd6_aqs', 'fnd6_beta',
       'fnd6_capxv', 'fnd6_ceql', 'fnd6_ch', 'fnd6_ci', 'fnd6_cibegni',
       'fnd6_cicurr', 'fnd6_cidergl', 'fnd6_cik', 'fnd6_cimii',
       'fnd6_ciother', 'fnd6_cipen', 'fnd6_cisecgl', 'fnd6_citotal',
       'fnd6_city', 'fnd6_cld2', 'fnd6_cld3', 'fnd6_cld4', 'fnd6_cld5',
       'fnd6_cptmfmq_actq', 'fnd6_cptmfmq_atq', 'fnd6_cptmfmq_ceqq',
       'fnd6_cptmfmq_dlttq', 'fnd6_cptmfmq_dpq', 'fnd6_cptmfmq_lctq',
       'fnd6_cptmfmq_oibdpq', 'fnd6_cptmfmq_o

### 将datafield替换到alpha模板中 group_rank({fundamental model data}/cap,subindustry),批量生产alpha

In [8]:
alpha_list = []

for datafield in datafields_list_fundamental6:
    print('正在將alpha表达式与setting封装')
    alpha_expression = f'group_rank({datafield}/cap,subindustry)'
    print(alpha_expression)
    simulation_data = {
    'type': 'REGULAR',
    'settings' :{
        'instrumentType':'EQUITY',
        'region':'USA',
        'universe': 'TOP3000',
        'delay' : 1,
        'decay' : 0,
        'neutralization' : 'SUBINDUSTRY',
        'truncation':  0.08,
        'pasteurization': 'ON',
        'unitHandling' : 'VERIFY',
        'nanHandling' : 'ON',
        'language' : 'FASTEXPR',
        'visualization': False,
        },
    'regular':alpha_expression
    }
    alpha_list.append(simulation_data)
print(f'一共封装了 {len(alpha_list)} 个alpha表达式')

正在將alpha表达式与setting封装
group_rank(assets/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(assets_curr/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(bookvalue_ps/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(capex/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cash/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cash_st/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cashflow/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cashflow_dividends/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cashflow_fin/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cashflow_invst/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cashflow_op/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(cogs/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(current_ratio/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(debt/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(debt_lt/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(debt_st/cap,subindustry)
正在將alpha表达式与setting封装
group_rank(depre_am

In [9]:
alpha_list[1]

{'type': 'REGULAR',
 'settings': {'instrumentType': 'EQUITY',
  'region': 'USA',
  'universe': 'TOP3000',
  'delay': 1,
  'decay': 0,
  'neutralization': 'SUBINDUSTRY',
  'truncation': 0.08,
  'pasteurization': 'ON',
  'unitHandling': 'VERIFY',
  'nanHandling': 'ON',
  'language': 'FASTEXPR',
  'visualization': False},
 'regular': 'group_rank(assets_curr/cap,subindustry)'}

### 一个个发到服务器回测

In [ ]:
from time import sleep
for alpha in alpha_list:
    sim_resp = sess.post(
        'https://api.worldquantbrain.com/simulations',
        json=alpha
    )
    try:
        sim_progress_url = sim_resp.headers.get('Location')
        while True:
            sim_progress_resp = sess.get(sim_progress_url)
            retry_after_sec = float(sim_progress_resp.headers.get('Retry-After', '0'))
            if retry_after_sec == 0:
                break
            sleep(retry_after_sec)
        alpha_id = sim_progress_resp.json()['alpha']
        print(f'Alpha ID: {alpha_id}')
    except :
        print(f'提交失败:等10秒后继续')
        sleep(2)

Alpha ID: 58VvZWQX
Alpha ID: MPAxjX76
Alpha ID: 58VvZkeM
Alpha ID: qMZXKYmA
Alpha ID: JjLdjlGW
Alpha ID: KPRLPkzk
Alpha ID: A1d31lrQ
Alpha ID: d5wQ5aKj
Alpha ID: WjmgjRgQ
Alpha ID: 6XbEmXwO
Alpha ID: O0v9QaEd
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
提交失败:等10秒后继续
